In [2]:
import os
import yaml
import copy
import pickle


import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from os import path as pt
from tqdm import tqdm
sns.set_style("darkgrid")  # 원하는 스타일 선택

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import *
from src.evaluation.summary import full_evaluation

In [ ]:
# # Load configuration dict
# config_dir = 'configs/config.yaml'
# with open(config_dir) as file:
#     config = ml_collections.ConfigDict(yaml.safe_load(file))

# if (config.device == "cuda" and torch.cuda.is_available()):
#     config.update({"device": "cuda:1"}, allow_val_change=True)
#     torch.cuda.set_device(0)
# else:
#     config.update({"device": "cpu"}, allow_val_change=True)

# set_seed(config.seed)

### Data Preprocess

In [6]:
def rolling_window(data, window_size):
    n_windows = data.shape[0] - window_size + 1
    windows = np.zeros((n_windows, window_size, data.shape[1]))
    for idx in range(n_windows):
        windows[idx] = data[idx:idx + window_size]
    return windows

# # Step 1: Load and preprocess data
# df = pd.read_csv(f"./data/indices.csv")
# df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
# df.set_index('Date', inplace=True)
# df = df.apply(pd.to_numeric).astype(float)

# # Step 2: Compute log returns
# log_returns = np.diff(np.log(df), axis=0)
# print(log_returns.shape)

# # Step 3: Scale the log returns
# log_returns_scaled, scalers = scaling(log_returns)

# # Step 4: Prepare initial prices and create rolling windows
# init_price = torch.from_numpy(np.array(df)[:-(128), :]).float().unsqueeze(1)
# log_returns_scaled = torch.from_numpy(rolling_window(log_returns_scaled, 128)).float()
# print('init_price:', init_price.shape)
# print('log_returns_scaled:', log_returns_scaled.shape)

In [7]:
# --- [0. 준비 단계] ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 256

# 데이터 로드
df = pd.read_csv(f"./data/indices.csv")
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df.set_index('Date', inplace=True)
df = df.apply(pd.to_numeric).astype(float)

log_returns = np.log1p(df.pct_change()).dropna().values
init_price_raw = np.array(df)[:-1, :]

# --- [1. Early Split (시간 순 분할)] ---
total_len = len(log_returns)
train_size = int(0.7 * total_len)

train_raw = log_returns[:train_size]
test_raw = log_returns[train_size:] # Test는 원본 6차원 그대로 대기

train_init_raw = init_price_raw[:train_size]
test_init_raw = init_price_raw[train_size:]

# --- [2. GARCH 12차원 전처리 (Train ONLY)] ---
# Train만 GARCH로 12차원 분해 및 스케일링 수행
train_12d, scalers = scaling(train_raw)


train_12d = train_12d[:train_size]

# --- [3. Windowing (시퀀스 쪼개기)] ---
# Train은 12차원 윈도우 생성
train_windowed = torch.from_numpy(rolling_window(train_12d, seq_len)).float()
train_init_windowed = torch.from_numpy(train_init_raw[:-(seq_len-1)]).float().unsqueeze(1)

test_windowed = torch.from_numpy(rolling_window(test_raw, seq_len)).float()
test_init_windowed = torch.from_numpy(test_init_raw[:-(seq_len-1)]).float().unsqueeze(1)

# (B, T, C) -> (B, C, T) 축 변환
training_data = train_windowed.clone().detach() # (B, 256, 12)
test_data = test_windowed.clone().detach()  # (B, 256, 6)

training_set = TensorDataset(training_data)
test_set = TensorDataset(test_data)

train_dl = DataLoader(training_set, batch_size=256, shuffle=False)
test_dl = DataLoader(test_set, batch_size=256, shuffle=False)

print("\n🚀 데이터 준비 완료!")
print(f"Train DataLoader 배치 크기: {next(iter(train_dl))[0].shape}")
print(f"Test DataLoader 배치 크기: {next(iter(test_dl))[0].shape}")

c:\Users\asas4\OneDrive\Dokumen\GitHub\quantDiffusion\scratch\QuantGAN\src\preprocess\gaussianize.py:179: UserWarning: Warning: No convergence after 100 iterations. Increase max_iter.
  warnings.warn("Warning: No convergence after %d iterations. Increase max_iter." % max_iter)



🚀 데이터 준비 완료!
Train DataLoader 배치 크기: torch.Size([256, 256, 6])
Test DataLoader 배치 크기: torch.Size([256, 256, 6])


c:\Users\asas4\OneDrive\Dokumen\GitHub\quantDiffusion\scratch\QuantGAN\src\preprocess\gaussianize.py:179: UserWarning: Warning: No convergence after 100 iterations. Increase max_iter.
  warnings.warn("Warning: No convergence after %d iterations. Increase max_iter." % max_iter)


### Generative models for time series generation

### Data Construction

We divide the data into training and validation set for the offline evaluation of our model

In [ ]:
perm_idx = torch.randperm(log_returns_scaled.shape[0])
train_size = int(0.7*log_returns_scaled.shape[0])

training_data = log_returns_scaled[perm_idx[:train_size]]
train_init_price = init_price[perm_idx[:train_size]]
test_data = log_returns_scaled[perm_idx[train_size:]]
test_init_price = init_price[perm_idx[train_size:]]

print("training_data: ", training_data.shape)
print("test_data: ", test_data.shape)

Load the dataset


In [8]:
config = {
    # Data-related settings
    'file_name': 'indices',
    'data_dir': 'data/',
    'result_dir': 'results/',
    'log_interval': 5,
    'n_vars': 6,
    'n_steps': 128,  # 실제 데이터 시퀀스 길이와 일치 (training_data: 128 스텝)

    # Training settings
    'device': 'cuda',
    'seed': 0,
    'batch_size': 256,
    'n_epochs': 300,

    # Algorithm-related settings
    'algo': 'QuantGAN',

    # Model settings
    'G_hidden_dim': 80,
    'D_hidden_dim': 80,
    'output_dim': 1,
    'noise_dim': 6,
    'clip': 0.01,

    # Learning rates
    'lr_G': 0.0002,
    'lr_D': 0.0001,
    'D_steps_per_G_step': 1
}

from types import SimpleNamespace

# 기존 config 딕셔너리를 객체 형식으로 변환
config = SimpleNamespace(**config)

In [ ]:
training_set = TensorDataset(training_data)
test_set = TensorDataset(test_data)

train_dl = DataLoader(training_set, batch_size=config.batch_size, shuffle=True)
test_dl = DataLoader(test_set, batch_size=config.batch_size, shuffle=True)

### Generative model

Here we construct a generator and a discriminator for this task. Both the generator and discriminator takes as input the time series. Then we have the training algorithm TailGANTrainer.

### Initialize the generator, discriminator and the trainer

In [9]:
from src.baselines.networks.discriminators import UserDiscriminator
from src.baselines.networks.generators import UserGenerator
from src.baselines.trainer import *

generator = UserGenerator(config)
discriminator = UserDiscriminator(config)
trainer = GANTrainer(G=generator, D=discriminator, train_dl=train_dl, config=config)

In [10]:
def count_parameters(model):
    # 학습 가능한(Trainable) 파라미터 수
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 전체(Total) 파라미터 수
    total_params = sum(p.numel() for p in model.parameters())
    
    print(f"✅ 전체 파라미터 수: {total_params:,}")
    print(f"✅ 학습 가능한 파라미터 수: {trainable_params:,}")
    return total_params

# 모델 인스턴스를 전달하여 실행
count_parameters(generator)

✅ 전체 파라미터 수: 165,046
✅ 학습 가능한 파라미터 수: 165,046


165046

### Model training and saving

In [ ]:
# Model training
trainer.fit()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data1 = torch.randn(256, 6, 128)
data2 = torch.randn(256, 6, 256)

print(data1.shape)
print(generator(data1.to(device)).shape)
print(data2.shape)
print(generator(data2.to(device)).shape)

# Model Evaluation

### MVFIT-GAN

In [11]:
from scipy.stats import entropy, wasserstein_distance, ks_2samp, spearmanr, kendalltau
from statsmodels.tsa.stattools import acf
from scipy.spatial.distance import jensenshannon

full_name = "indices_300_256_G_lr_0.0002_D_lr_0.0001_noise_6_hidden_dim_80"

def compute_avg_emd(real_data, fake_data, window):
    """
    Compute the Earth Mover's Distance (EMD) for real and fake data over a rolling window.
    """
    emd = 0
    for i in range(real_data.shape[1]):  # Iterate over features
        real_dist = rolling_window(real_data[:, i, :].T, window).sum(axis=1).ravel()
        fake_dist = rolling_window(fake_data[:, i, :].T, window).sum(axis=1).ravel()
        emd += wasserstein_distance(real_dist, fake_dist)
    return emd

window = 100

In [14]:
""" Find the best epochs based on 100 days cumulative distribution """

gen_tmp = UserGenerator(config)
min_dist = float('inf')  
best_epoch = 0

batch_size = 1000
for epoch in range(50, 300, 5):
    
    # Load generator for the current epoch
    generator.load_state_dict(torch.load(f'./results/models/{full_name}/Generator_{epoch}.pt'))
    generator.to(config.device)
    generator.eval()
    
    # Generate fake data
    noise = torch.randn(batch_size, config.noise_dim, config.n_steps, device=config.device)
    with torch.no_grad():        
        fake= generator(noise)                
        
    # Inverse scaling for real and fake data
    print(fake.shape)
    print(training_data.transpose(1, 2).shape)
    fake_data = inverse_scaling(fake, scalers)        
    real_data = inverse_scaling(training_data.transpose(1, 2), scalers)
    
    # Compute EMD
    emd = compute_avg_emd(real_data, fake_data, window)

    # Update best epoch if current EMD is lower
    if emd < min_dist:
        min_dist = emd
        best_epoch = epoch
        print(f"min_distance: {min_dist:.3f}, best_epoch: {best_epoch}") 

generator.load_state_dict(torch.load(f'./results/models/{full_name}/Generator_{best_epoch}.pt'))
generator.to(config.device)
generator.eval()
print()

torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 518995171.200, best_epoch: 50
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 3282.199, best_epoch: 60
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 76.798, best_epoch: 70
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 2.333, best_epoch: 75
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 2.173, best_epoch: 90
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 0.502, best_epoch: 95
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 0.486, best_epoch: 100
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 256])
min_distance: 0.339, best_epoch: 105
torch.Size([1000, 6, 128])
torch.Size([3748, 6, 25

In [15]:
generator.load_state_dict(torch.load(f'./results/models/{full_name}/Generator_150.pt'))
""" Fake data와 Test data를 비교 """
batch_size = 2000
noise = torch.randn(batch_size, config.noise_dim, config.n_steps, device=config.device)
with torch.no_grad():
    fake = generator(noise)

fake_data = inverse_scaling(fake, scalers)
real_data = inverse_scaling(training_data.transpose(1, 2), scalers)
emd = compute_avg_emd(real_data, fake_data, window)
print(emd)

0.201506250235705


In [16]:
def rolling_window(data, window_size):
    n_windows = data.shape[0] - window_size + 1
    windows = np.zeros((n_windows, window_size, data.shape[1]))
    for idx in range(n_windows):
        windows[idx] = data[idx:idx + window_size]
    return windows

In [17]:
import gc
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import norm, kstest # KS 검정을 위해 kstest 추가

from statsmodels.tsa.stattools import acf
from scipy.stats import skew, kurtosis, wasserstein_distance, ks_2samp




def rolling_window(data, window_size):
    n_windows = data.shape[0] - window_size + 1
    windows = np.zeros((n_windows, window_size, data.shape[1]))
    for idx in range(n_windows):
        windows[idx] = data[idx:idx + window_size]
    return windows


def _scalar_diff_loss(diff, loss_type):
    """Compute scalar loss from diff array: mse, mae, or frobenius."""
    if loss_type == 'mse':
        return np.mean(diff ** 2)
    if loss_type == 'mae':
        return np.mean(np.abs(diff))
    if loss_type in ('frobenius', 'fro'):
        return np.linalg.norm(diff)
    raise ValueError(f"Unsupported loss_type: {loss_type}")


def correlation_mean(data):
    """
    각 샘플의 상관행렬을 계산한 후, 그 평균을 반환.
    
    Parameters:
        data (np.ndarray): shape = (num_samples, num_time_steps, num_features)
    
    Returns:
        mean_correlation_matrix (np.ndarray): shape = (num_features, num_features)
    """
    num_samples = data.shape[0]
    correlations = []

    for i in range(num_samples):
        sample = data[i]  # shape = (num_time_steps, num_features)
        corr_mat = np.corrcoef(sample, rowvar=False)
        correlations.append(corr_mat)

    mean_correlation_matrix = np.mean(correlations, axis=0)  
    return mean_correlation_matrix


def ccf_mean_vectorized(data, max_lag=10):
    """
    행렬 연산(Vectorization)을 활용하여 매우 빠르고 메모리 효율적인 CCF 평균 계산
    - data: shape = (num_samples, time_steps, num_features)
    """
    B, T, F = data.shape

    # 1. 시간 축(axis=1)을 기준으로 전체 평균과 표준편차 계산
    mu = np.mean(data, axis=1, keepdims=True)  # (B, 1, F)
    # 모집단 표준편차(ddof=0) 사용: statsmodels의 acovf 기본값과 일치
    std = np.std(data, axis=1, keepdims=True)  # (B, 1, F)

    # 2. Z-Score 정규화 (표준편차가 0인 경우 0으로 나누는 것을 방지하기 위해 np.divide 사용)
    Z = np.divide(data - mu, std, out=np.zeros_like(data), where=std != 0)

    # 3. 유효한(valid) 데이터 마스크 생성 (std가 0이 아닌 정상 샘플 필터링)
    valid_mask = (std[:, 0, :] != 0)  # (B, F)
    # i번째 feature와 j번째 feature 둘 다 유효해야 하므로 (B, F, F) 마스크 생성
    valid_pair_mask = valid_mask[:, :, None] & valid_mask[:, None, :]  

    # 4. 결과를 저장할 배열: (F, F, max_lag + 1)
    mean_ccf = np.zeros((F, F, max_lag + 1))

    # 5. 각 시차(lag)별로 행렬곱 연산 수행 (for문은 단 max_lag+1 번만 돎)
    for k in range(max_lag + 1):
        if k == 0:
            Z_0 = Z
            Z_k = Z
        else:
            Z_0 = Z[:, :-k, :]  # 0 ~ (T-k-1)
            Z_k = Z[:, k:, :]   # k ~ (T-1)

        # einsum을 이용해 배치(b) 및 시간(t) 축에 대해 3차원 행렬을 한 번에 내적(Dot product)
        # Z_0: (B, T-k, F), Z_k: (B, T-k, F) -> 결과: (B, F, F)
        # statsmodels.ccf(adjusted=False)의 공식에 맞춰 분모를 전체 길이 T로 나눔
        ccf_k_batch = np.einsum('bti, btj -> bij', Z_0, Z_k) / T

        # 비정상 샘플(표준편차 0이라 연산 불가)의 결과는 제외하기 위해 마스킹 처리
        ccf_k_batch_valid = np.where(valid_pair_mask, ccf_k_batch, 0)

        # 배치 차원(axis=0)에 대해 합산 및 유효 개수 카운트
        sum_ccf = np.sum(ccf_k_batch_valid, axis=0)    # (F, F)
        count_valid = np.sum(valid_pair_mask, axis=0)  # (F, F)

        # 평균 계산 (유효 샘플이 0개인 구간은 NaN 방지를 위해 0으로 채움)
        mean_ccf[:, :, k] = np.divide(sum_ccf, count_valid, out=np.zeros_like(sum_ccf, dtype=float), where=count_valid != 0)

    return mean_ccf


def _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='mse'):
    """
    (기존 사용하시던 로직 유지. _scalar_diff_loss 함수가 정의되어 있다고 가정합니다.)
    """
    diff = fake_mean_ccf - real_mean_ccf
    # 사용자의 환경에 맞게 오차 계산 (아래는 예시)
    if loss_type == 'mae':
        loss_val = np.mean(np.abs(diff))
    elif loss_type == 'mse':
        loss_val = np.mean(diff ** 2)
    elif loss_type == 'frobenius':
        loss_val = np.linalg.norm(diff)
    else:
        # 기존에 _scalar_diff_loss를 호출하셨다면 해당 함수 사용
        loss_val = _scalar_diff_loss(diff, loss_type) 
        
    return round(float(loss_val), 4)


def calculate_ccf(real_data, fake_data, max_lag):
    CCF_dict = {}

    # CCF 계산을 위해 데이터 전치 후, 백터화 함수 직접 호출
    real_mean_ccf = ccf_mean_vectorized(np.transpose(real_data, (0, 2, 1)), max_lag=max_lag)
    fake_mean_ccf = ccf_mean_vectorized(np.transpose(fake_data, (0, 2, 1)), max_lag=max_lag)

    # 손실 계산
    loss_mae = _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='mae')
    loss_frob = _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='frobenius')
    loss_mse = _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='mse')

    if max_lag == 0:
        CCF_dict[f'CCF(τ = 0)_MAE'] = loss_mae
        CCF_dict[f'CCF(τ = 0)_FROB'] = loss_frob
        CCF_dict[f'CCF(τ = 0)_MSE'] = loss_mse
    
    else:
        CCF_dict[f'CCF(τ > 0)_MAE'] = loss_mae
        CCF_dict[f'CCF(τ > 0)_FROB'] = loss_frob
        CCF_dict[f'CCF(τ > 0)_MSE'] = loss_mse

    df_avg_scores = pd.DataFrame(list(CCF_dict.values()), index=CCF_dict.keys(), columns=['Average'])
    df_avg_scores.index.name = 'Metric'

    return df_avg_scores


def calculate_distribution_scores(real, fake, windows):
    """
    real: (B_real, companys, T)
    fake: (B_fake, companys, T)
    """
    # 1. 배열 차원 확보 (2D가 들어와도 3D로 안전하게 처리)
    real = np.atleast_3d(real)
    fake = np.atleast_3d(fake)
    
    # B는 다를 수 있으므로 각각 추출, companys(C)와 T는 동일하다고 가정
    B_r, C, T_r = real.shape
    B_f, _, T_f = fake.shape
    
    results = {}
    metrics = ['EMD', 'KS']
    
    for w in windows:
        # 1. O(1) 롤링 윈도우 연산 (시간 축인 axis=2 기준 누적합)
        real_cs = np.insert(np.cumsum(real, axis=2), 0, 0, axis=2)
        fake_cs = np.insert(np.cumsum(fake, axis=2), 0, 0, axis=2)
        
        real_roll = real_cs[:, :, w:] - real_cs[:, :, :-w]  # (B_r, C, T - w + 1)
        fake_roll = fake_cs[:, :, w:] - fake_cs[:, :, :-w]  # (B_f, C, T - w + 1)
        
        # ==========================================================
        # 3. EMD & KS Statistic 연산 (종목별 풀링 분포 거리 측정)
        # ==========================================================
        # B가 다르므로 1:1 매칭 불가 -> 종목(C)별로 모든 배치의 데이터를 하나로 풀링
        # np.swapaxes를 통해 (C, B, T_w)로 바꾼 뒤 평탄화 -> Shape: (C, B * T_w)
        real_pooled = np.swapaxes(real_roll, 0, 1).reshape(C, -1)
        fake_pooled = np.swapaxes(fake_roll, 0, 1).reshape(C, -1)
        
        emd_scores = np.zeros(C)
        ks_scores = np.zeros(C)
        
        # 각 종목(Company) 단위로 완성된 두 분포의 거리를 측정 (for문 C번만 실행됨)
        for c in range(C):
            emd_scores[c] = wasserstein_distance(real_pooled[c], fake_pooled[c])
            ks_scores[c], _ = ks_2samp(real_pooled[c], fake_pooled[c])
            
        # ==========================================================
        # 4. 결과 저장
        # ==========================================================
        results[f'EMD({w})'] = np.round(np.mean(emd_scores), decimals=4)
        results[f'KS({w})'] = np.round(np.mean(ks_scores), decimals=4)
    # DataFrame 포맷팅
    ordered_keys = [f"{m}({w})" for m in metrics for w in windows]
    df_avg_scores = pd.DataFrame([results[k] for k in ordered_keys], index=ordered_keys, columns=['Average'])
    df_avg_scores.index.name = 'Metric'
    
    return df_avg_scores

def calculate_skew_kurtosis(real_data, fake_data, windows):
    """
    각 샘플(배치)별 Skew/Kurt를 계산하여 평균 차이를 구하고, 
    요청하신 단일 컬럼 DataFrame 형식으로 반환합니다.
    - real_data: (B_real, C, T)
    - fake_data: (B_fake, C, T)
    """
    B_r, C, T_r = real_data.shape
    B_f, _, T_f = fake_data.shape
    
    # 내부에서 자산별 리스트 생성 (제시해주신 입력 방식 반영)
    real_list = [real_data[:, i, :] for i in range(C)]
    fake_list = [fake_data[:, i, :] for i in range(C)]
    
    results = {}
    metrics = ['Skew', 'Kurt'] # EMD, KS는 외부에서 합친다고 가정하거나 여기서는 Skew/Kurt만 처리

    for w in windows:
        skew_diffs_assets = []
        kurt_diffs_assets = []

        for i in range(C):
            real_arr = np.asarray(real_list[i])   # (B_r, T)
            fake_arr = np.asarray(fake_list[i])   # (B_f, T)

            # 배치 사이즈가 다를 경우 최소값에 맞춤
            num = min(real_arr.shape[0], fake_arr.shape[0])
            r_arr = real_arr[:num]
            f_arr = fake_arr[:num]

            # 롤링 윈도우 합산 (CumSum 활용 최적화)
            r_cs = np.pad(np.cumsum(r_arr, axis=1), ((0, 0), (1, 0)), mode="constant")
            f_cs = np.pad(np.cumsum(f_arr, axis=1), ((0, 0), (1, 0)), mode="constant")

            r_roll = r_cs[:, w:] - r_cs[:, :-w]
            f_roll = f_cs[:, w:] - f_cs[:, :-w]

            # 배치별 skew / kurtosis 계산 (axis=1)
            r_skews = skew(r_roll, axis=1, bias=True, nan_policy="omit")
            f_skews = skew(f_roll, axis=1, bias=True, nan_policy="omit")

            r_kurts = kurtosis(r_roll, axis=1, fisher=False, bias=True, nan_policy="omit")
            f_kurts = kurtosis(f_roll, axis=1, fisher=False, bias=True, nan_policy="omit")

            # 샘플 평균값의 차이(Absolute Error of Means)
            skew_diff = np.abs(np.nanmean(r_skews) - np.nanmean(f_skews))
            kurt_diff = np.abs(np.nanmean(r_kurts) - np.nanmean(f_kurts))

            skew_diffs_assets.append(skew_diff)
            kurt_diffs_assets.append(kurt_diff)

        # 모든 자산에 대한 평균값을 결과 딕셔너리에 저장
        results[f'Skew({w})'] = np.round(np.nanmean(skew_diffs_assets), decimals=4)
        results[f'Kurt({w})'] = np.round(np.nanmean(kurt_diffs_assets), decimals=4)

    # DataFrame 포맷팅 (요청하신 형식)
    ordered_keys = [f"{m}({w})" for m in metrics for w in windows]
    df_avg_scores = pd.DataFrame(
        [results[k] for k in ordered_keys], 
        index=ordered_keys, 
        columns=['Average']
    )
    df_avg_scores.index.name = 'Metric'
    
    return df_avg_scores


def vectorized_batch_acf_3d(data, nlags=30):
    """
    (B, C, T) 형태의 3차원 데이터를 받아 모든 배치와 종목의 ACF를 한 번에 계산
    - data: shape = (B, C, T)
    - return: shape = (B, C, nlags + 1)
    """
    data = np.atleast_3d(data)
    B, C, T = data.shape
    
    # 시간 축(axis=2)을 기준으로 평균 차감
    mu = np.mean(data, axis=2, keepdims=True)
    x = data - mu
    
    # c_0 (분산) - ACF의 분모
    c0 = np.sum(x**2, axis=2) / T
    
    # 결과를 담을 3차원 배열 (lag 0은 항상 1.0)
    acf_res = np.zeros((B, C, nlags + 1))
    acf_res[:, :, 0] = 1.0  
    
    # 시차(lag)별로 행렬 슬라이싱 내적 연산 수행
    for lag in range(1, nlags + 1):
        # 시간 축(axis=2)을 lag만큼 이동시킨 두 배열 간의 내적 (c_k)
        ck = np.sum(x[:, :, :-lag] * x[:, :, lag:], axis=2) / T
        
        # 0으로 나누는 오류 방지
        acf_res[:, :, lag] = np.divide(ck, c0, out=np.zeros((B, C)), where=c0 != 0)
        
    return acf_res


def calculate_acf_score(
    real,
    fake,
    lags=30,
    loss_type='mse'
):
    """
    real: (B_r, C, T)
    fake: (B_f, C, T)
    """
    real = np.atleast_3d(real)
    fake = np.atleast_3d(fake)
    
    B_r, C, T_r = real.shape
    B_f, _, T_f = fake.shape

    titles = ['ACF(x)', 'ACF(|x|)', 'ACF(x^2)']
    data_transforms = [lambda x: x, np.abs, np.square]  
    
    # 결과를 저장할 딕셔너리 초기화
    acf_scores = {f"Group {i+1}": {} for i in range(C)}
    avg_scores_per_transform = {title: [] for title in titles}

    for transform, title in zip(data_transforms, titles):
        # (1) 3차원 데이터 전체 변환 (Identity, Absolute, Squared)
        t_real = transform(real)
        t_fake = transform(fake)
        
        # (2) 초고속 3D ACF 병렬 계산 -> shape: (B, C, lags + 1)
        acf_real = vectorized_batch_acf_3d(t_real, nlags=lags)
        acf_fake = vectorized_batch_acf_3d(t_fake, nlags=lags)
        
        # (3) 배치 축(axis=0)에 대한 평균 ACF 도출 -> shape: (C, lags + 1)
        # 배치(B) 사이즈가 서로 달라도 이 단계에서 대푯값으로 압축되므로 문제없이 비교 가능합니다.
        mean_acf_real = np.mean(acf_real, axis=0)
        mean_acf_fake = np.mean(acf_fake, axis=0)
        
        # (4) Real 평균 ACF vs Fake 평균 ACF 오차 계산 -> shape: (C, lags + 1)
        diff = mean_acf_fake - mean_acf_real
        
        # (5) 종목(C)별로 Loss 계산
        for i in range(C):
            # _scalar_diff_loss가 1D 배열(lags + 1 길이)을 받는다고 가정
            val = round(float(_scalar_diff_loss(diff[i], loss_type)), 4)
            acf_scores[f"Group {i+1}"][title] = val
            avg_scores_per_transform[title].append(val)
            
    # (6) 전체 "평균의 평균(Mean of Means)" 계산
    overall_average_scores = {
        title: round(np.mean(vals), 4) for title, vals in avg_scores_per_transform.items()
    }

    # (7) DataFrame 포맷팅
    df_overall = pd.DataFrame(list(overall_average_scores.values()), 
                              index=overall_average_scores.keys(), 
                              columns=['Average'])
    df_overall.index.name = 'Metric'

    return acf_scores, df_overall



def vectorized_batch_leverage_effect_3d(data, lags=30):
    """
    (B, C, T) 형태의 3차원 데이터를 받아 모든 배치와 종목의 레버리지 효과를 한 번에 계산
    - data: shape = (B, C, T)
    - return: shape = (B, C, lags)
    """
    data = np.atleast_3d(data)
    B, C, T = data.shape
    
    # 결과를 담을 3차원 배열
    leverage_res = np.zeros((B, C, lags))
    
    # tau(시차)별로 슬라이싱하여 모든 샘플, 모든 종목의 상관계수를 행렬 내적으로 한 번에 계산
    for tau in range(1, lags + 1):
        # 시간 축(axis=2)을 기준으로 슬라이싱
        # X: r_t (앞부분), Y: r_{t+tau}^2 (뒷부분의 제곱)
        X = data[:, :, :-tau]
        Y = data[:, :, tau:]**2
        
        # 시간 축(axis=2) 기준 평균 차감 (Z-score 분자 부분)
        X_c = X - np.mean(X, axis=2, keepdims=True)
        Y_c = Y - np.mean(Y, axis=2, keepdims=True)
        
        # 상관계수 분자 (공분산 계산 시 분모인 N-1은 약분되므로 생략 가능)
        cov_num = np.sum(X_c * Y_c, axis=2)
        
        # 상관계수 분모 (분산 계산 시 분모인 N-1은 약분됨)
        var_X = np.sum(X_c**2, axis=2)
        var_Y = np.sum(Y_c**2, axis=2)
        denom = np.sqrt(var_X * var_Y)
        
        # 0으로 나누는 오류 방지
        leverage_res[:, :, tau - 1] = np.divide(cov_num, denom, out=np.zeros((B, C)), where=denom != 0)
        
    return leverage_res


def calculate_leverage_effect(real, fake, lags=30, loss_type='mse'):
    """
    real: (B_r, C, T)
    fake: (B_f, C, T)
    """
    real = np.atleast_3d(real)
    fake = np.atleast_3d(fake)
    
    B_r, C, T_r = real.shape
    B_f, _, T_f = fake.shape

    leverage_scores = {}
    all_group_losses = []

    # (1) 3D 벡터화를 통한 초고속 병렬 계산 -> shape: (B_r, C, lags) & (B_f, C, lags)
    real_leverage = vectorized_batch_leverage_effect_3d(real, lags=lags)
    fake_leverage = vectorized_batch_leverage_effect_3d(fake, lags=lags)

    # (2) 배치 축(axis=0)에 대한 평균 도출 -> shape: (C, lags)
    # 배치 크기(B_r, B_f)가 달라도 여기서 종목별 대푯값으로 압축되므로 1:1 비교가 가능해집니다.
    mean_real_leverage = np.mean(real_leverage, axis=0)
    mean_fake_leverage = np.mean(fake_leverage, axis=0)

    # (3) Real과 Fake 평균 간의 차이 계산 -> shape: (C, lags)
    diff = mean_fake_leverage - mean_real_leverage

    # (4) 종목(C)별로 Loss 계산
    for i in range(C):
        # _scalar_diff_loss가 1D 배열(lags 길이)을 받는다고 가정
        loss_val = round(float(_scalar_diff_loss(diff[i], loss_type)), 4)
        leverage_scores[f'Group {i+1}'] = loss_val
        all_group_losses.append(loss_val)

    # (5) 전체 평균 손실 계산
    overall_mean = round(float(np.mean(all_group_losses)), 4)

    # (6) DataFrame 래핑
    df_overall = pd.DataFrame(
        [overall_mean], 
        index=['Leverage Effect'], 
        columns=['Average']
    )
    df_overall.index.name = 'Metric'

    return leverage_scores, df_overall

# ---------------------------------------------------------
# [내부 헬퍼 함수] 시각화 함수 내부에서 쓸 초고속 연산기
# ---------------------------------------------------------
def _fast_acf_2d(data, nlags):
    """시각화용 초고속 2D ACF 계산기"""
    data = np.atleast_2d(data)
    N, T = data.shape
    mu = np.mean(data, axis=1, keepdims=True)
    x = data - mu
    c0 = np.sum(x**2, axis=1) / T
    
    res = np.zeros((N, nlags + 1))
    res[:, 0] = 1.0
    for lag in range(1, nlags + 1):
        ck = np.sum(x[:, :-lag] * x[:, lag:], axis=1) / T
        res[:, lag] = np.divide(ck, c0, out=np.zeros(N), where=c0 != 0)
    return res


In [19]:
test_data.shape

torch.Size([1462, 256, 6])

In [18]:
real_data = test_data.transpose(1, 2)
# real_data = torch.from_numpy(rolling_window(train_raw, seq_len)).float().transpose(1, 2)
if hasattr(real_data, 'cpu'):
    real_data = real_data.cpu().detach().numpy()
total_result_df=pd.DataFrame()
for seed in range(5):

    """ Fake data와 Test data를 비교 """
    batch_size = 2000
    noise = torch.randn(batch_size, config.noise_dim, config.n_steps, device=config.device)
    with torch.no_grad():
        fake = generator(noise)

    fake_data = inverse_scaling(fake, scalers)
    print(fake_data.shape)
    fake_list = [fake_data[:, i, :] for i in range(fake_data.shape[1])]
    real_list = [real_data[:, i, :] for i in range(real_data.shape[1])]


    """ Visualize the distribution of the real and fake data """

    # Plot the distribution of the real and fake data
    windows = [1, 5, 20, 100]

    result_df=pd.DataFrame()
    CCF_df = calculate_ccf(real_data, fake_data, 0)
    result_df = pd.concat([result_df, CCF_df], axis=0)
    total_CCF_df=pd.DataFrame(columns=['Average'])
    for i in range(1,11):
        total_CCF_df = pd.concat([total_CCF_df, calculate_ccf(real_data, fake_data, i)], axis=1)

    result_df = pd.concat([result_df, pd.DataFrame(total_CCF_df.mean(axis=1), columns=['Average'])], axis=0)

    similarity_df = calculate_distribution_scores(real_data, fake_data, windows)
    result_df = pd.concat([result_df, similarity_df], axis=0)
    skew_kurt_df = calculate_skew_kurtosis(real_data, fake_data, windows)
    result_df = pd.concat([result_df, skew_kurt_df], axis=0)
    _, acf_df = calculate_acf_score(real_data, fake_data, 40)
    result_df = pd.concat([result_df, acf_df], axis=0)
    _, leverage_df = calculate_leverage_effect(real_data, fake_data, 40)
    result_df = pd.concat([result_df, leverage_df], axis=0)

    total_result_df = pd.concat([total_result_df, result_df], axis=1)

statistics_df = pd.DataFrame(index=total_result_df.index, columns=['MEAN', 'STD'])
statistics_df.loc[statistics_df.index, 'MEAN']=total_result_df.mean(axis=1).values
statistics_df.loc[statistics_df.index, 'STD']=total_result_df.std(axis=1).values
print(statistics_df)

(2000, 6, 128)
(2000, 6, 128)
(2000, 6, 128)
(2000, 6, 128)
(2000, 6, 128)
                     MEAN       STD
CCF(τ = 0)_MAE    0.13358  0.000356
CCF(τ = 0)_FROB   0.98194  0.002592
CCF(τ = 0)_MSE     0.0268  0.000158
CCF(τ > 0)_MAE   0.091878  0.000194
CCF(τ > 0)_FROB  1.700916  0.003057
CCF(τ > 0)_MSE   0.014576  0.000055
EMD(1)             0.0019       0.0
EMD(5)            0.00612  0.000045
EMD(20)           0.01536  0.000114
EMD(100)          0.04608  0.000773
KS(1)             0.04802  0.000148
KS(5)             0.07352  0.000342
KS(20)            0.11244  0.001101
KS(100)           0.18784  0.002251
Skew(1)            0.3061  0.012925
Skew(5)           0.28382  0.004674
Skew(20)           0.3024  0.003317
Skew(100)         0.06398  0.003033
Kurt(1)           2.22048  0.020705
Kurt(5)           1.24592  0.007645
Kurt(20)          0.94132  0.005442
Kurt(100)          0.1417  0.008457
ACF(x)             0.0029       0.0
ACF(|x|)           0.0057       0.0
ACF(x^2)           0.0032

In [ ]:


eval_size = real_data.shape[0]

fake_data_torch = torch.from_numpy(fake_data).float().transpose(1, 2)
real_data_torch = torch.from_numpy(real_data).float().transpose(1, 2)

test_init_price = train_init_price
test_init_price[:] = 100

# 1. 원본 데이터를 보존하기 위해 clone()을 사용하고, 100으로 초기화합니다.
# (test_init_price = train_init_price 라고 하면 원본 값도 100으로 바뀝니다)
test_init_price = train_init_price.clone()
test_init_price[:] = 100

# 2. 평가할 실제 데이터의 양을 정확히 계산합니다.
# fake_data_torch가 eval_size보다 작을 경우를 대비합니다.
actual_eval_data = fake_data_torch[:eval_size]
n_samples = actual_eval_data.shape[0]

# 3. 데이터 개수(n_samples)에 맞춰서 초기 가격도 똑같이 잘라줍니다.
fake_prices = log_return_to_price(actual_eval_data, test_init_price[:n_samples])
real_prices = log_return_to_price(real_data_torch[:n_samples], test_init_price[:n_samples])


config_dir = f'{LOCAL_BASE_DIRECTORY}/utils/eval_portfolio/config.yaml'
with open(config_dir) as file:
    eval_config = ml_collections.ConfigDict(yaml.safe_load(file))


all_positive = (fake_prices > 0).all()
if not all_positive:
    raise ValueError("Sanity Check Failed: Some fake prices are not positive.")

res_dict = {
    "var_abs_mean": 0.,
    "var_rel_mean": 0.,
    "es_abs_mean": 0.,
    "es_rel_mean": 0.,
}


# 1. 설정 및 변수 준비
alphas = eval_config.alphas  # [0.05, 0.025, 0.01]
strat_names = ['equal_weight', 'mean_reversion', 'trend_following', 'vol_trading']
# 표시용 약어 매핑
strat_map = {
    'equal_weight': 'EW',
    'mean_reversion': 'MR',
    'trend_following': 'TF',
    'vol_trading': 'VT'
}

# 결과를 저장할 중첩 딕셔너리 {Metric(alpha): {Strategy: Value}}
final_results = {}

with torch.no_grad():
    for alpha in alphas:
        # Config의 alpha 설정 동기화
        eval_config.Evaluation.TestMetrics.var.alpha = alpha
        eval_config.Evaluation.TestMetrics.es.alpha = alpha
        
        # 각 Metric별로 전략들의 수치를 담을 임시 딕셔너리
        var_row = {}
        es_row = {}

        for strat_name in strat_names:
            # 전략별 평가 수행
            subres_dict, _, _ = full_evaluation(
                fake_prices, 
                real_prices, 
                eval_config, 
                strat_name=strat_name
            )
            
            # 약어 이름으로 개별 전략 결과 저장
            short_name = strat_map[strat_name]
            var_row[short_name] = round(subres_dict.get('var_rel_mean', 0.), 4)
            es_row[short_name] = round(subres_dict.get('es_rel_mean', 0.), 4)
            
        # 4개 전략의 평균(Average) 계산 및 추가
        var_row['Average'] = round(np.mean(list(var_row.values())), 4)
        es_row['Average'] = round(np.mean(list(es_row.values())), 4)
        
        # 최종 결과 사전에 인덱스 명칭으로 저장
        final_results[f"RE_VaR({alpha})"] = var_row
        final_results[f"RE_ES({alpha})"] = es_row
        

# 2. DataFrame 포맷팅
# Metric 순서 정의 (VaR 그룹 후 ES 그룹)
metrics = ['RE_VaR', 'RE_ES']
ordered_keys = [f"{m}({a})" for m in metrics for a in alphas]

# 딕셔너리로부터 DataFrame 생성 및 컬럼 순서 정렬
df_final = pd.DataFrame.from_dict(final_results, orient='index')
cols_order = [strat_map[s] for s in strat_names] + ['Average']
df_final = df_final.reindex(index=ordered_keys, columns=cols_order)
df_final.index.name = 'Metric'

In [ ]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# ---------------------------------------------------------
# 1. 로그 수익률 -> 가격 궤적 재구성 (B, C, T 맞춤형)
# ---------------------------------------------------------
def reconstruct_prices(log_returns, s0=100.0):
    """
    초기 자산 가격을 100으로 설정하고 누적합을 지수화하여 시장 시나리오 재구성
    - log_returns shape: (B, C, T)
    """
    log_returns = np.atleast_3d(log_returns)
    B, C, T = log_returns.shape
    
    # t=0 시점의 가격 100을 포함하기 위해 시간축(T)을 T+1 크기로 생성
    prices = np.zeros((B, C, T + 1))
    prices[:, :, 0] = s0
    
    # 시간축(axis=2) 기준으로 누적합(CumSum) 적용
    prices[:, :, 1:] = s0 * np.exp(np.cumsum(log_returns, axis=2))
    return prices

# ---------------------------------------------------------
# 2. 포트폴리오 전략 함수 (B, C, T 맞춤형)
# ---------------------------------------------------------
def calculate_portfolio_returns(prices, window=20):
    B, C, T_plus_1 = prices.shape
    T = T_plus_1 - 1
    
    # 단순 수익률 계산 (P_t / P_{t-1} - 1) -> Shape: (B, C, T)
    R = prices[:, :, 1:] / prices[:, :, :-1] - 1.0  

    port_ret = {}

    # 1. EW (Equal Weight): 종목축(axis=1) 기준으로 평균 내어 포트폴리오 수익률 도출 -> Shape: (B, T)
    port_ret['EW'] = np.mean(R, axis=1)

    # 2. TF & 3. MR
    port_ret['TF'] = np.zeros((B, T))
    port_ret['MR'] = np.zeros((B, T))
    
    # t-1 시점의 수익률 부호(Sign) -> Shape: (B, C, T-1)
    sign_prev = np.sign(R[:, :, :-1])  
    R_current = R[:, :, 1:]            
    
    # 종목별 가중치를 수익률에 곱한 뒤 합산(axis=1)
    port_ret['TF'][:, 1:] = np.sum((sign_prev / C) * R_current, axis=1)
    port_ret['MR'][:, 1:] = np.sum((-sign_prev / C) * R_current, axis=1)

    # 4. VT (Volatility Trading)
    port_ret['VT'] = np.zeros((B, T))
    actual_window = min(window, max(1, T - 1)) # 시계열이 윈도우보다 짧을 때 에러 방지
    
    if T > actual_window:
        R_past = R[:, :, :-1]
        
        # 시간축(axis=2) 기준으로 슬라이딩 윈도우 투영
        windows = sliding_window_view(R_past, window_shape=actual_window, axis=2) 
        roll_std = np.std(windows, axis=-1, ddof=0) + 1e-8 

        # 변동성에 역산 비례하도록 비중 조절
        inv_std = 1.0 / roll_std
        weights = inv_std / np.sum(inv_std, axis=1, keepdims=True) 

        R_target = R[:, :, actual_window:] 
        port_ret['VT'][:, actual_window:] = np.sum(weights * R_target, axis=1)

    return port_ret, actual_window

# ---------------------------------------------------------
# 3. 꼬리 위험(Tail Risk) 측정
# ---------------------------------------------------------
def get_var_es_from_distribution(pnl_distribution, alpha):
    """
    1차원으로 평탄화된 P&L 누적 분포 배열에서 VaR와 ES 산출
    """
    # 에러 방지용 Safety Check (배열이 비어있을 경우 0 반환)
    if pnl_distribution.size == 0:
        return 0.0, 0.0
        
    var = np.percentile(pnl_distribution, alpha * 100)
    mask = pnl_distribution <= var
    
    if np.sum(mask) == 0:
        es = 0.0
    else:
        es = np.mean(pnl_distribution[mask])
        
    return var, es

# ---------------------------------------------------------
# 4. 메인 파이프라인
# ---------------------------------------------------------
alphas = [0.05, 0.025, 0.01]
strategies = ['EW', 'TF', 'MR', 'VT']
window_size = 100

print("📊 논문 구현: 포트폴리오 전략별 누적 P&L 기반 RE (Relative Error)")
print("-" * 65)

# Step 1: 가격 재구성
real_prices = reconstruct_prices(real_data, s0=100.0)
fake_prices = reconstruct_prices(fake_data, s0=100.0)

# Step 2: 전략별 포트폴리오 수익률 도출
real_port_ret, win_r = calculate_portfolio_returns(real_prices, window=window_size)
fake_port_ret, win_f = calculate_portfolio_returns(fake_prices, window=window_size)

# 실제와 생성 데이터 중 안전하게 겹치는 웜업(Warm-up) 기간 선택
valid_window = max(win_r, win_f)

results_dict = {}

for alpha in alphas:
    row_var = {}
    row_es = {}
    
    for strat in strategies:
        # Step 3: 웜업 기간 제외 -> Shape: (Batch, 유효한 Time)
        valid_real_ret = real_port_ret[strat][:, valid_window:]
        valid_fake_ret = fake_port_ret[strat][:, valid_window:]
        
        # Step 4: 논문 핵심! 모든 중간 시점의 P&L 누적(CumSum) 분포 구성 및 평탄화
        real_cum_pnl_dist = np.cumsum(valid_real_ret, axis=1).flatten()
        fake_cum_pnl_dist = np.cumsum(valid_fake_ret, axis=1).flatten()
        
        # Step 5: VaR, ES 추출
        var_r, es_r = get_var_es_from_distribution(real_cum_pnl_dist, alpha)
        var_f, es_f = get_var_es_from_distribution(fake_cum_pnl_dist, alpha)
        
        # Step 6: 상대 오차 계산
        eps = 1e-8
        re_var = np.abs(var_f - var_r) / (np.abs(var_r) + eps)
        re_es = np.abs(es_f - es_r) / (np.abs(es_r) + eps)
        
        row_var[strat] = round(float(re_var), 4)
        row_es[strat] = round(float(re_es), 4)
        
    # 각 alpha 별 전략들의 평균값 산출
    row_var['Average'] = round(float(np.mean(list(row_var.values()))), 4)
    row_es['Average'] = round(float(np.mean(list(row_es.values()))), 4)
    
    results_dict[f'RE_VaR_{alpha}'] = row_var
    results_dict[f'RE_ES_{alpha}'] = row_es

# Step 7: DataFrame 포맷팅
ordered_keys = [f'RE_VaR_{a}' for a in alphas] + [f'RE_ES_{a}' for a in alphas]
df_tail_risk = pd.DataFrame.from_dict(results_dict, orient='index')
df_tail_risk = df_tail_risk[strategies + ['Average']]
df_tail_risk = df_tail_risk.reindex(ordered_keys)
df_tail_risk.index.name = 'Metric'

print(df_tail_risk.T)
print("-" * 65)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, ks_2samp, skew, kurtosis

def calculate_distribution_scores(real, fake, windows):
    """
    real: (B_real, companys, T)
    fake: (B_fake, companys, T)
    """
    # 1. 배열 차원 확보 (2D가 들어와도 3D로 안전하게 처리)
    real = np.atleast_3d(real)
    fake = np.atleast_3d(fake)
    
    # B는 다를 수 있으므로 각각 추출, companys(C)와 T는 동일하다고 가정
    B_r, C, T_r = real.shape
    B_f, _, T_f = fake.shape
    
    results = {}
    metrics = ['EMD', 'KS', 'Skew', 'Kurt']
    
    for w in windows:
        # 1. O(1) 롤링 윈도우 연산 (시간 축인 axis=2 기준 누적합)
        real_cs = np.insert(np.cumsum(real, axis=2), 0, 0, axis=2)
        fake_cs = np.insert(np.cumsum(fake, axis=2), 0, 0, axis=2)
        
        real_roll = real_cs[:, :, w:] - real_cs[:, :, :-w]  # (B_r, C, T - w + 1)
        fake_roll = fake_cs[:, :, w:] - fake_cs[:, :, :-w]  # (B_f, C, T - w + 1)
        
        # ==========================================================
        # 2. Skewness & Kurtosis 연산 (배치별 선(先) 계산 후 평균 차이)
        # ==========================================================
        # 시간 축(axis=2)에 대해 각 샘플의 왜도/첨도 도출 -> Shape: (B, C)
        real_skew = skew(real_roll, axis=2)
        fake_skew = skew(fake_roll, axis=2)
        real_kurt = kurtosis(real_roll, axis=2)
        fake_kurt = kurtosis(fake_roll, axis=2)
        
        # 배치 축(axis=0)에 대해 평균을 내어 종목별(C) 대푯값 도출 -> Shape: (C,)
        mean_real_skew = np.mean(real_skew, axis=0)
        mean_fake_skew = np.mean(fake_skew, axis=0)
        mean_real_kurt = np.mean(real_kurt, axis=0)
        mean_fake_kurt = np.mean(fake_kurt, axis=0)
        
        # 종목별 차이의 절댓값을 구한 뒤, 모든 종목(C)에 대해 최종 평균 -> Scalar
        skew_diff = np.mean(np.abs(mean_real_skew - mean_fake_skew))
        kurt_diff = np.mean(np.abs(mean_real_kurt - mean_fake_kurt))
        
        # ==========================================================
        # 3. EMD & KS Statistic 연산 (종목별 풀링 분포 거리 측정)
        # ==========================================================
        # B가 다르므로 1:1 매칭 불가 -> 종목(C)별로 모든 배치의 데이터를 하나로 풀링
        # np.swapaxes를 통해 (C, B, T_w)로 바꾼 뒤 평탄화 -> Shape: (C, B * T_w)
        real_pooled = np.swapaxes(real_roll, 0, 1).reshape(C, -1)
        fake_pooled = np.swapaxes(fake_roll, 0, 1).reshape(C, -1)
        
        emd_scores = np.zeros(C)
        ks_scores = np.zeros(C)
        
        # 각 종목(Company) 단위로 완성된 두 분포의 거리를 측정 (for문 C번만 실행됨)
        for c in range(C):
            emd_scores[c] = wasserstein_distance(real_pooled[c], fake_pooled[c])
            ks_scores[c], _ = ks_2samp(real_pooled[c], fake_pooled[c])
            
        # ==========================================================
        # 4. 결과 저장
        # ==========================================================
        results[f'EMD({w})'] = np.round(np.mean(emd_scores), decimals=4)
        results[f'KS({w})'] = np.round(np.mean(ks_scores), decimals=4)
        results[f'Skew({w})'] = np.round(skew_diff, decimals=4)
        results[f'Kurt({w})'] = np.round(kurt_diff, decimals=4)

    # DataFrame 포맷팅
    ordered_keys = [f"{m}({w})" for m in metrics for w in windows]
    df_avg_scores = pd.DataFrame([results[k] for k in ordered_keys], index=ordered_keys, columns=['Average'])
    df_avg_scores.index.name = 'Metric'
    
    return df_avg_scores

In [ ]:
# fake_data=fake_data2
""" Visualize the distribution of the real and fake data """
fake_list = [fake_data[:, i, :] for i in range(fake_data.shape[1])]
real_list = [real_data[:, i, :] for i in range(real_data.shape[1])]

In [ ]:
similarity_df = calculate_distribution_scores(real_data, fake_data, [1,5,20,100])

In [ ]:
similarity_df

In [ ]:
similarity_df

In [ ]:
x = fake_data.transpose(0, 2, 1)
result = x.reshape(-1, x.shape[-1])

y = real_data.transpose(0, 2, 1)
result2 = y.reshape(-1, y.shape[-1])
print(result.shape) # (1536, 128) 예상

import matplotlib as mpl

mpl.rcdefaults()
plt.figure(figsize=(12, 3))
plt.hist(result[:,1], bins=1000, alpha=0.5, density=True, label='quantDiffusion scaling')
plt.hist(result2[:,1], bins=1000, alpha=0.5, density=True, label='After scaling')
plt.legend()
plt.show()

In [ ]:
real_skew = skew(result)
fake_skew = skew(result2)
skew_diff = np.abs(real_skew - fake_skew)

real_kurt = kurtosis(result)
fake_kurt = kurtosis(result2)
kurt_diff = np.abs(real_kurt - fake_kurt)

In [ ]:
skew_diff

In [ ]:
kurt_diff

In [ ]:
similarity_df

In [ ]:
log_returns = np.log1p(df.pct_change()).dropna().values
log_returns_windowed = torch.from_numpy(rolling_window(log_returns, 128)).float()
log_returns_scaled, scalers = scaling(log_returns)
log_returns_scaled_windowed = torch.from_numpy(rolling_window(log_returns_scaled, 128)).float()
log_returns_inversed = inverse_scaling(torch.tensor(log_returns_scaled_windowed.permute(0, 2, 1)), scalers)
log_returns_inversed_windowed = torch.from_numpy(log_returns_inversed.transpose(0, 2, 1)).float()

In [ ]:
fake.permute(0,2,1).shape

In [ ]:
import gc
samples_inversed = inverse_scaling(fake, scalers)
samples_inversed_windowed = torch.from_numpy(samples_inversed.transpose(0, 2, 1)).float()
for i in range(6):
    col = i
    plt.hist(log_returns_windowed[0,:,col], bins=100, alpha=0.5, density=True, label='Before scaling')
    plt.hist(log_returns_inversed_windowed[0,:,col], bins=100, alpha=0.5, density=True, label='After inverse scaling')
    plt.hist(samples_inversed_windowed[0,:,col], bins=100, alpha=0.5, density=True, label='quantDiffusion inverse scaling')
    plt.legend()
    plt.show()
    gc.collect()

for i in range(6):
    col = i
    plt.hist(log_returns_scaled_windowed[0,:,col], bins=100, alpha=0.5, density=True, label='After scaling')
    plt.hist(fake.permute(0,2,1)[0,:,col].cpu(), bins=100, alpha=0.5, density=True, label='quantDiffusion scaling')
    plt.legend()
    plt.show()
    gc.collect()

In [ ]:
# 각 feature의 전체 최소값 및 최대값 계산
min_vals = real_data.min(axis=(0, 2), keepdims=True)  # Shape: (1, 5, 1)
max_vals = real_data.max(axis=(0, 2), keepdims=True)  # Shape: (1, 5, 1)

# 마스크 계산 (모든 샘플이 범위 내에 있는지 확인)
mask = np.all((fake_data >= min_vals * 1.5) & (fake_data <= max_vals * 1.5), axis=(1, 2))

# 마스크를 적용하여 fake_data 필터링
print(real_data.shape, type(real_data))
print(fake_data.shape, type(fake_data))
fake_data2 = fake_data[mask]
fake_data2 = fake_data[:real_data.shape[0]]
print(f"Filtered shape: {fake_data2.shape}")

In [ ]:
x = fake_data2.transpose(0, 2, 1)
result = x.reshape(-1, x.shape[-1])

y = real_data.transpose(0, 2, 1)
result2 = y.reshape(-1, y.shape[-1])
print(result.shape) # (1536, 128) 예상

import matplotlib as mpl

mpl.rcdefaults()
plt.figure(figsize=(12, 3))
plt.hist(result[:,1], bins=1000, alpha=0.5, density=True, label='quantDiffusion scaling')
plt.hist(result2[:,1], bins=1000, alpha=0.5, density=True, label='After scaling')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

def calculate_var_es(returns, alpha=0.05):
    """수익률 array에 대한 VaR 및 ES 계산"""
    var = -np.percentile(returns, alpha * 100)
    tail_losses = returns[returns <= -var]
    es = -tail_losses.mean() if len(tail_losses) > 0 else var
    return var, es

def calculate_detailed_re_se(real_arr, fake_arr, alpha=0.05):
    """
    각 변수(Asset)별로 RE와 SE를 각각 산출
    real_arr, fake_arr: (n_samples, n_vars)
    """
    # 2차원 유지 확인
    if real_arr.ndim == 1: real_arr = real_arr.reshape(-1, 1)
    if fake_arr.ndim == 1: fake_arr = fake_arr.reshape(-1, 1)
    
    n_samples, n_vars = real_arr.shape
    
    re_per_asset = []
    se_per_asset = []

    for i in range(n_vars):
        # i번째 변수(종목) 추출
        real_ret = real_arr[:, i]
        fake_ret = fake_arr[:, i]
        
        # 1. 지표 계산 (Real, Fake, Sampled)
        var_r, es_r = calculate_var_es(real_ret, alpha)
        var_g, es_g = calculate_var_es(fake_ret, alpha)
        
        # SE용 랜덤 샘플링
        sample_indices = np.random.choice(len(real_ret), len(fake_ret), replace=True)
        var_s, es_s = calculate_var_es(real_ret[sample_indices], alpha)

        # 2. 개별 변수의 RE/SE 계산 (공식: 1/2 * (|ΔVaR|/VaR + |ΔES|/ES))
        asset_re = abs(var_g - var_r) / abs(var_r)
        asset_se = 0.5 * (abs(var_s - var_r) / abs(var_r) + abs(es_s - es_r) / abs(es_r))
        
        re_per_asset.append(asset_re)
        se_per_asset.append(asset_se)

    # 결과를 데이터프레임으로 정리하여 가독성 높임
    result_df = pd.DataFrame({
        'Asset_Index': range(n_vars),
        'RE': re_per_asset,
        'SE': se_per_asset
    })
    
    # 이미지에 나온 전체 평균값도 함께 계산
    total_re = np.mean(re_per_asset)
    total_se = np.mean(se_per_asset)

    return result_df, total_re, total_se

# --- 실행 ---
# result[:, 0] 형태의 (5000, 6) 데이터 입력
df_errors, avg_re, avg_se = calculate_detailed_re_se(result, result2, alpha=0.05)

print("📊 [종목별 상세 에러 보고서]")
print(df_errors.to_string(index=False))
print(f"\n📢 전체 평균 RE: {avg_re:.6f}")
print(f"📢 전체 평균 SE: {avg_se:.6f}")

In [ ]:
# fake_data=fake_data2
""" Visualize the distribution of the real and fake data """
fake_list = [fake_data[:, i, :] for i in range(fake_data.shape[1])]
real_list = [real_data[:, i, :] for i in range(real_data.shape[1])]

# Plot the distribution of the real and fake data
windows = [1, 20, 100]
for j in range(config.n_vars):
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(18, 4))  

    for i in range(len(windows)):
        col = i

        real_dist = rolling_window(real_list[j].T, windows[i]).sum(axis=1).ravel()
        fake_dist = rolling_window(fake_list[j].T, windows[i]).sum(axis=1).ravel()        
        
        min_val = real_dist.min()
        max_val = real_dist.max()
        
        bins = np.linspace(min_val, max_val, 81)  
        
        sns.histplot(real_dist, bins=bins, kde=False, ax=axs[col], color='tab:blue', alpha=0.5, stat='density')
        sns.histplot(fake_dist, bins=bins, kde=False, ax=axs[col], color='tab:orange', alpha=0.5, stat='density')

        axs[col].set_xlim(*np.quantile(real_dist, [0.001, .999]))
        
        axs[col].set_title('{} day return distribution'.format(windows[i]), size=18)
        axs[col].yaxis.grid(True, alpha=0.5)
        axs[col].set_xlabel('Cumulative log return', fontsize=12)
        axs[col].set_ylabel('Frequency', fontsize=12)

    axs[0].legend(['Historical returns', 'Synthetic returns'])

plt.show()

def calculate_distribution_scores(real, fake, num_G, windows):
    scores = {
        'EMD': np.zeros((num_G, len(windows))),
        'KL': np.zeros((num_G, len(windows))),
        'JS': np.zeros((num_G, len(windows))),
        'KS': np.zeros((num_G, len(windows)))
    }

    for i in range(num_G):
        for j in range(len(windows)):
            real_dist = rolling_window(real[i].T, windows[j]).sum(axis=1).ravel()
            fake_dist = rolling_window(fake[i].T, windows[j]).sum(axis=1).ravel()
            
            np.random.shuffle(real_dist)
            np.random.shuffle(fake_dist)
            
            # Calculate EMD
            scores['EMD'][i, j] = wasserstein_distance(real_dist, fake_dist)
            
            # Calculate KS Statistic
            scores['KS'][i, j], _ = ks_2samp(real_dist, fake_dist)                                    
            
            # Create histograms to estimate the probability distributions
            real_hist, bin_edges = np.histogram(real_dist, bins=100, density=True)
            fake_hist, _ = np.histogram(fake_dist, bins=bin_edges, density=True)
            
            # Normalize the histograms to get probability distributions
            real_prob = real_hist / np.sum(real_hist)
            fake_prob = fake_hist / np.sum(fake_hist)
            
            # Calculate KL Divergence
            kl_divergence = entropy(real_prob + 1e-10, fake_prob + 1e-10)
            scores['KL'][i, j] = kl_divergence

            # Calculate JS Divergence
            js_divergence = jensenshannon(real_prob + 1e-10, fake_prob + 1e-10)
            scores['JS'][i, j] = js_divergence
                
    df_scores = {}
    for metric, data in scores.items():
        data = np.round(data, decimals=4)
        df_scores[metric] = pd.DataFrame(data.T, index=windows, columns=[f'{metric} {i}' for i in range(num_G)])
        
    emd_avg = np.mean(scores['EMD'], axis=0)
    df_scores['EMD']['EMD_avg'] = np.round(emd_avg, decimals=4)
    
    return df_scores

# Calculate the distribution scores
windows = pd.Series([1, 5, 20, 100], name='window size')

results_emd = calculate_distribution_scores(real_list, fake_list, config.n_vars, windows)
results_emd

In [ ]:
from scipy.stats import skew, kurtosis

def calculate_skew_kurt_diff(fake_list, real_list):
    results = []
    skewness_diffs = []
    kurtosis_diffs = []
    
    for asset_idx in range(len(real_list)):
        fake_data = fake_list[asset_idx]
        real_data = real_list[asset_idx]

        # 각 샘플에 대해 skewness와 kurtosis를 계산하고 평균을 구합니다.
        fake_skewness = np.mean([skew(sample) for sample in fake_data])
        fake_kurtosis = np.mean([kurtosis(sample, fisher=False) for sample in fake_data])
        print(fake_kurtosis)
        real_skewness = np.mean([skew(sample) for sample in real_data])
        real_kurtosis = np.mean([kurtosis(sample, fisher=False) for sample in real_data])
        print(real_kurtosis)

        # skewness와 kurtosis의 차이 계산
        skewness_diff = np.abs(real_skewness - fake_skewness)
        kurtosis_diff = np.abs(real_kurtosis - fake_kurtosis)

        # 소수점 자릿수 제한
        skewness_diff = round(skewness_diff, 4)
        kurtosis_diff = round(kurtosis_diff, 4)

        results.append((skewness_diff, kurtosis_diff))
        skewness_diffs.append(skewness_diff)
        kurtosis_diffs.append(kurtosis_diff)

        print(f'Asset {asset_idx + 1} Skewness Difference: {skewness_diff}')
        print(f'Asset {asset_idx + 1} Kurtosis Difference: {kurtosis_diff}')
        
    # 평균 계산 수정
    avg_skewness_diff = round(np.mean(skewness_diffs), 4)
    avg_kurtosis_diff = round(np.mean(kurtosis_diffs), 4)

    print(f"\n[평균] Skewness Difference: {avg_skewness_diff}")
    print(f"[평균] Kurtosis Difference: {avg_kurtosis_diff}")
    
    return results

calculate_skew_kurt_diff(fake_list, real_list)

In [ ]:
""" visualizing acf plots """
def plot_acf_comparison(real_list, fake_list, n_vars, lags=30):    
    data_types = ['Identity', 'Squared']
    data_transforms = [lambda x: x, np.square]  
    titles = ['Identity log returns', 'Squared log returns']

    for i in range(n_vars):
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))  
        print(i)
        for ax, data_type, transform, title in zip(axs, data_types, data_transforms, titles):
            
            transformed_real = transform(real_list[i])
            transformed_fake = transform(fake_list[i])
            
            acf_real = np.array([acf(ts, nlags=lags) for ts in transformed_real])
            acf_fake = np.array([acf(ts, nlags=lags) for ts in transformed_fake])
            
            mean_real = np.mean(acf_real, axis=0)
            std_real = np.std(acf_real, axis=0)
            mean_fake = np.mean(acf_fake, axis=0)
            std_fake = np.std(acf_fake, axis=0)
                        
            ax.plot(mean_real, label=f'{data_type} ACF Real - Mean', color='tab:blue')
            ax.fill_between(range(lags+1), mean_real - 0.5*std_real, mean_real + 0.5*std_real, color='tab:blue', alpha=0.2,
                            label=f'{data_type} ACF Real - 1/2 Std Dev')        
            ax.plot(mean_fake, label=f'{data_type} ACF Synthetic - Mean', color='tab:orange')
            ax.fill_between(range(lags+1), mean_fake - 0.5*std_fake, mean_fake + 0.5*std_fake, color='tab:orange', alpha=0.2,
                            label=f'{data_type} ACF Synthetic - 1/2 Std Dev')
            
            ax.set_ylim(-0.15, 0.3)
            ax.set_title(title)
            ax.grid(True)
            ax.axhline(y=0, color='k')
            ax.axvline(x=0, color='k')
            ax.set_xlabel('Lag (number of days)', fontsize=16)
            
        axs[0].legend()

        plt.show()

plot_acf_comparison(real_list, fake_list, config.n_vars, lags=30)

import numpy as np
from statsmodels.tsa.stattools import acf

def calculate_acf_score_mean_vs_mean_no_std(
    real_list,
    fake_list,
    lags=30,
    loss_type='mse'
):
    """
    real_list[i], fake_list[i] 각각 하나의 "그룹" 또는 "자산" 집합이라고 가정
    각 real_list[i]에는 (n_real_samples_i, ) 형태의 시계열이 여러 개 들어있다고 보고,
    fake_list[i]에도 여러 시계열이 들어있다고 봄.

    -> Real 평균 ACF vs. Fake 평균 ACF만을 비교하여 MSE, MAE 등을 계산하는 버전.
       표준편차( std )는 따로 계산하지 않음.
    """
    # 변환 함수 정의
    data_transforms = [lambda x: x, np.abs, np.square]  
    titles = ['Identity log returns', 'Absolute log returns', 'Squared log returns']

    acf_scores = {}  # 그룹별 결과 저장
    # 최종적으로 transform별 "평균 점수(mean of means)"를 계산할 목적
    avg_scores_per_transform = {title: [] for title in titles}

    n_vars = len(real_list)
    for group_idx in range(n_vars):
        group_scores = {}  # 이 그룹에 대한 transform별 점수(스칼라값)

        for transform, title in zip(data_transforms, titles):
            # (1) Real / Fake 데이터 변환
            transformed_real = transform(real_list[group_idx])
            transformed_fake = transform(fake_list[group_idx])
            
            # (2) 각 샘플별 ACF 계산
            acf_real = np.array([acf(ts, nlags=lags) for ts in transformed_real])  
            acf_fake = np.array([acf(ts, nlags=lags) for ts in transformed_fake])  

            # (3) Real, Fake 각각의 평균 ACF 계산
            mean_acf_real = np.mean(acf_real, axis=0)  # shape=(lags+1,)
            mean_acf_fake = np.mean(acf_fake, axis=0)  # shape=(lags+1,)

            # (4) Real 평균 ACF vs. Fake 평균 ACF 차이
            diff = mean_acf_fake - mean_acf_real

            # (5) loss_type에 따라 계산 (MSE or MAE 등)
            if loss_type == 'mse':
                val = np.mean(diff**2)
            elif loss_type == 'mae':
                val = np.mean(np.abs(diff))
            else:
                raise ValueError(f"Unknown loss_type: {loss_type}")

            # (6) 표준편차는 사용하지 않으므로, 결과는 스칼라 val만
            group_scores[title] = round(val, 4)

            # (7) 전체 평균 계산을 위해 저장
            avg_scores_per_transform[title].append(val)

        # 해당 그룹에 대한 transform별 점수 기록
        acf_scores[f"Group {group_idx+1}"] = group_scores
    
    # 모든 그룹에 대한 “평균의 평균”(mean of means) 계산
    overall_average_scores = {}
    for title in titles:
        mean_of_means = np.mean(avg_scores_per_transform[title])
        overall_average_scores[title] = round(mean_of_means, 4)

    return acf_scores, overall_average_scores


# 사용 예시
lags = 30
loss_type = 'mse'  # 'mse', 'mae' 등

acf_scores, overall_avg_scores = calculate_acf_score_mean_vs_mean_no_std(
    real_list, 
    fake_list, 
    lags=lags, 
    loss_type=loss_type
)

print("=== ACF Scores (per group) ===")
for group_name, transform_scores in acf_scores.items():
    print(f"{group_name}:")
    for transform_title, val in transform_scores.items():
        print(f"  - {transform_title}: loss={val}")

print("\n=== Overall Average Scores ===")
for transform_title, mean_of_means in overall_avg_scores.items():
    print(f"{transform_title}: mean of means={mean_of_means}")


In [ ]:
def correlation_mean(data):
    """
    각 샘플의 상관행렬을 계산한 후, 그 평균을 반환.
    
    Parameters:
        data (np.ndarray): shape = (num_samples, num_time_steps, num_features)
    
    Returns:
        mean_correlation_matrix (np.ndarray): shape = (num_features, num_features)
    """
    num_samples = data.shape[0]
    correlations = []

    for i in range(num_samples):
        sample = data[i]  # shape = (num_time_steps, num_features)
        corr_mat = np.corrcoef(sample, rowvar=False)
        correlations.append(corr_mat)

    mean_correlation_matrix = np.mean(correlations, axis=0)  
    return mean_correlation_matrix


def correlation_loss(real_mean_corr, fake_mean_corr, loss_type='mse'):
    """
    Real과 Fake의 평균 상관행렬 간의 손실을 계산.
    
    Parameters:
        real_mean_corr (np.ndarray): Real 데이터의 평균 상관행렬
        fake_mean_corr (np.ndarray): Fake 데이터의 평균 상관행렬
        loss_type (str): 'mse', 'mae', 'frobenius' 중 선택
    
    Returns:
        loss (float): 지정된 손실 타입에 따른 손실 값 (소수점 4자리로 반올림)
    """
    diff = fake_mean_corr - real_mean_corr

    if loss_type == 'mse':
        loss_val = np.mean(diff**2)
    elif loss_type == 'mae':
        loss_val = np.mean(np.abs(diff))
    elif loss_type == 'frobenius':            
        loss_val = np.linalg.norm(diff, ord='fro')
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    return round(loss_val, 4)

# 상관행렬 계산을 위해 데이터 전치 (shape: (num_samples, num_time_steps, num_features))
real_mean_corr = correlation_mean(np.transpose(real_data, (0, 2, 1)))
fake_mean_corr = correlation_mean(np.transpose(fake_data, (0, 2, 1)))

# 손실 계산
loss_mae = correlation_loss(real_mean_corr, fake_mean_corr, loss_type='mae')
loss_frob = correlation_loss(real_mean_corr, fake_mean_corr, loss_type='frobenius')
loss_mse = correlation_loss(real_mean_corr, fake_mean_corr, loss_type='mse')

# 결과 출력
print("[Correlation Loss - MAE]")
print(f"Loss: {loss_mae}")

print("\n[Correlation Loss - Frobenius]")
print(f"Loss: {loss_frob}")

print("\n[Correlation Loss - MSE]")
print(f"Loss: {loss_mse}")


def plot_correlation_heatmap(correlation_matrix, title, feature_names):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(correlation_matrix, annot=True, cmap='cubehelix_r', linewidths=.5, ax=ax, fmt=".2f", annot_kws={"size": 12}, vmin=-0.1, vmax=1)
    
    # 축 이름 설정
    ax.set_xticklabels(feature_names, fontsize=12)
    ax.set_yticklabels(feature_names, fontsize=12)
    
    # 제목 및 시각적 요소
    ax.set_title(title, fontsize=15)
    ax.tick_params(axis='both', which='major', labelsize=14)
    plt.tight_layout()
    plt.show()

feature_names = ['DJI', 'IXIC', 'JPM', 'HSI', 'Gold', 'WTI']
plot_correlation_heatmap(real_mean_corr, "Real Data", feature_names)
plot_correlation_heatmap(fake_mean_corr, "Fake Data", feature_names)


In [ ]:
import numpy as np
from statsmodels.tsa.stattools import ccf

def ccf_single_sample(x, max_lag=10):
    """
    x: shape = (time_steps, features)
    - statsmodels.tsa.stattools.ccf(x[:, i], x[:, j])를 이용해
      i, j 모든 쌍에 대해 lag=0..max_lag 범위의 cross-correlation을 구한다.
    - 결과를 3D 배열 (features, features, max_lag+1)에 저장하여 반환.

    Returns
    -------
    cc_matrix: np.ndarray
      shape = (num_features, num_features, max_lag+1)
      cc_matrix[i, j, k] = ccf 결과에서 lag=k인 상관계수
    """
    time_steps, num_features = x.shape
    cc_matrix = np.zeros((num_features, num_features, max_lag+1), dtype=float)

    for i in range(num_features):
        for j in range(num_features):
            # statsmodels ccf: lag >= 0 에 대한 상관계수
            c_array = ccf(x[:, i], x[:, j], adjusted=False)
            # lag=0 ~ max_lag만 추출
            c_array = c_array[:(max_lag+1)]
            # ccf 함수는 무한대로 확장될 수 있으므로, 필요한 max_lag+1까지만 자르기
            if len(c_array) < (max_lag+1):
                # 부족한 경우 0으로 채우기
                c_array = np.pad(c_array, (0, max_lag+1 - len(c_array)), 'constant')
            cc_matrix[i, j, :] = c_array

    return cc_matrix

def ccf_mean(data, max_lag=10):
    """
    data: shape = (num_samples, time_steps, features)

    각 샘플마다 ccf_single_sample -> (features, features, max_lag+1)
    모든 샘플에 대해 평균낸 결과를 반환.

    Returns
    -------
    mean_ccf: np.ndarray
      shape = (num_features, num_features, max_lag+1)
    """
    num_samples, time_steps, num_features = data.shape

    # 모든 샘플의 CCF 결과를 쌓아두기
    all_cc = np.zeros((num_samples, num_features, num_features, max_lag+1), dtype=float)

    for s_idx in range(num_samples):
        cc_matrix = ccf_single_sample(data[s_idx], max_lag=max_lag)
        all_cc[s_idx] = cc_matrix  # shape=(features, features, max_lag+1)

    # 샘플 차원에 대해 평균
    mean_ccf = all_cc.mean(axis=0)  # shape=(features, features, max_lag+1)
    return mean_ccf

def ccf_loss_mean_vs_mean(real_mean_ccf, fake_mean_ccf, loss_type='mse'):
    """
    - real_mean_ccf: shape = (features, features, max_lag+1)
      Real 데이터의 평균 CCF 행렬
    - fake_mean_ccf: shape = (features, features, max_lag+1)
      Fake 데이터의 평균 CCF 행렬
    - loss_type: 'mse', 'mae', 'frobenius' 중 선택

    returns: loss (float), 소수점 4자리로 반올림
    """
    # 두 평균 CCF 행렬의 차이
    diff = fake_mean_ccf - real_mean_ccf

    # 손실 계산
    if loss_type == 'mse':
        loss_val = np.mean(diff**2)
    elif loss_type == 'mae':
        loss_val = np.mean(np.abs(diff))
    elif loss_type == 'frobenius':
        # 3D 배열에 대한 Frobenius norm
        loss_val = np.linalg.norm(diff)
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    return round(loss_val, 4)

# 사용 예시
max_lag = 10

# CCF 계산을 위해 데이터 전치 (shape: (num_samples, time_steps, num_features))
# 기존에 사용된 np.transpose(real_data, (0, 2, 1))은 이미 올바른 형상으로 가정
real_mean_ccf = ccf_mean(np.transpose(real_data, (0, 2, 1)), max_lag=max_lag)
fake_mean_ccf = ccf_mean(np.transpose(fake_data, (0, 2, 1)), max_lag=max_lag)

# 손실 계산
loss_mae = ccf_loss_mean_vs_mean(real_mean_ccf, fake_mean_ccf, loss_type='mae')
loss_frob = ccf_loss_mean_vs_mean(real_mean_ccf, fake_mean_ccf, loss_type='frobenius')
loss_mse = ccf_loss_mean_vs_mean(real_mean_ccf, fake_mean_ccf, loss_type='mse')

# 결과 출력
print("[Partial CCF Loss - MAE]")
print(f"Loss: {loss_mae}")

print("\n[Partial CCF Loss - Frobenius]")
print(f"Loss: {loss_frob}")

print("\n[Partial CCF Loss - MSE]")
print(f"Loss: {loss_mse}")


In [ ]:
def leverage_effect_loss_mean_vs_mean(real_list, fake_list, lags=30, loss_type='mse'):
    """
    Real 데이터의 평균 레버리지 효과와 Fake 데이터의 평균 레버리지 효과를 비교하여 손실을 계산.

    Parameters:
        real_list (list of np.ndarray): 각 그룹별 Real 데이터. 각 요소는 shape=(n_real_samples, time_length)
        fake_list (list of np.ndarray): 각 그룹별 Fake 데이터. 각 요소는 shape=(n_fake_samples, time_length)
        lags (int): 최대 시차 (tau) 값
        loss_type (str): 손실 함수의 종류 ('mse', 'mae', 'fro')

    Returns:
        leverage_scores (dict): 그룹별 손실 값 저장
        overall_mean (float): 모든 그룹의 평균 손실
    """

    def leverage_effect(ts, tau):
        """
        레버리지 효과 계산:
        Corr(r_t, r_(t+tau)^2)
        """
        rt = ts[:-tau]            # 뒤에서 tau 개 제외
        rt_squared = ts[tau:]**2  # 앞에서 tau 개 제외한 뒤 제곱
        return np.corrcoef(rt, rt_squared)[0, 1]  # 상관계수

    n_vars = len(real_list)
    leverage_scores = {}     # 그룹별 손실 값 저장
    all_group_losses = []    # 모든 그룹의 손실 값을 모아 전체 평균 손실 계산

    for i in range(n_vars):
        real_data = real_list[i]  # shape=(n_real_samples, time_length)
        fake_data = fake_list[i]  # shape=(n_fake_samples, time_length)

        # (1) Real 각 샘플의 레버리지 효과 (tau=1부터 lags까지)
        real_leverage_effects = np.array([
            [leverage_effect(ts, tau) for tau in range(1, lags+1)]
            for ts in real_data
        ])  # shape = (n_real_samples, lags)

        # (2) Fake 각 샘플의 레버리지 효과
        fake_leverage_effects = np.array([
            [leverage_effect(ts, tau) for tau in range(1, lags+1)]
            for ts in fake_data
        ])  # shape = (n_fake_samples, lags)

        # (3) Real의 평균 레버리지 효과 계산
        mean_real_leverage = np.mean(real_leverage_effects, axis=0)  # shape=(lags,)

        # (4) Fake의 평균 레버리지 효과 계산
        mean_fake_leverage = np.mean(fake_leverage_effects, axis=0)  # shape=(lags,)

        # (5) Real 평균과 Fake 평균 간의 차이 계산
        diff = mean_fake_leverage - mean_real_leverage

        # (6) loss_type에 따라 손실 계산
        if loss_type == 'mse':
            loss_val = np.mean(diff**2)
        elif loss_type == 'mae':
            loss_val = np.mean(np.abs(diff))
        elif loss_type == 'frobenius':
            loss_val = np.linalg.norm(diff)
        else:
            raise ValueError(f"Unsupported loss_type: {loss_type}")

        # (7) 손실 값 저장 (소수점 반올림은 최종 저장 시점에 적용)
        leverage_scores[f'Group {i+1}'] = loss_val
        all_group_losses.append(loss_val)

        # (8) 그룹별 손실 출력 (소수점 반올림)
        print(f"[Group {i+1}] Leverage Loss ({loss_type.upper()}) => {loss_val:.4f}")

    # (9) 전체 그룹의 평균 손실 계산
    overall_mean = np.mean(all_group_losses)
    print(f"\n[Overall] Leverage Loss ({loss_type.upper()}) => {overall_mean:.4f}")

    # (10) 최종 결과를 소수점 4자리로 반올림하여 저장
    for group in leverage_scores:
        leverage_scores[group] = round(leverage_scores[group], 4)
    overall_mean = round(overall_mean, 4)

    return leverage_scores, overall_mean

# 예시 사용

# 손실 계산 (예: 'mse', 'mae', 'frobenius' 중 선택)
loss_type = 'mse'  # 'mse', 'mae', 'frobenius'

leverage_scores, overall_mean = leverage_effect_loss_mean_vs_mean(
    real_list, 
    fake_list, 
    lags=lags, 
    loss_type=loss_type
)

# 최종 결과 출력
print("\n=== Leverage Effect Loss (Mean vs. Mean) ===")
for group, loss in leverage_scores.items():
    print(f"{group}: {loss}")
print(f"Overall Mean Loss: {overall_mean}")

### Synthetic data generation

### Model evaluation

We compute the performance of our model by first generating the price process, apply the prespecified trading strategies and compare the resulting PnL process using the real and fake data.

In [ ]:
from src.evaluation.strategies import log_return_to_price

eval_size = real_data.shape[0]
fake_data_torch = torch.from_numpy(fake_data).float().transpose(1, 2)
real_data_torch = torch.from_numpy(real_data).float().transpose(1, 2)

fake_prices = log_return_to_price(fake_data_torch[:eval_size], train_init_price[:eval_size])
real_prices = log_return_to_price(real_data_torch[:eval_size], train_init_price[:eval_size])

In [ ]:
config_dir = 'src/evaluation/config.yaml'
with open(config_dir) as file:
    eval_config = ml_collections.ConfigDict(yaml.safe_load(file))
    
print(fake_data_torch.shape, real_data_torch.shape, test_init_price.shape)

all_positive = (fake_prices > 0).all()
if not all_positive:
    raise ValueError("Sanity Check Failed: Some fake prices are not positive.")

res_dict = {"var_mean" : 0., "es_mean": 0., "max_drawback_mean": 0., "cumulative_pnl_mean": 0.,}

# Do final evaluation
num_strat = 4

with torch.no_grad():
    for strat_name in ['equal_weight', 'mean_reversion', 'trend_following', 'vol_trading']:
        subres_dict = full_evaluation(fake_prices, real_prices, eval_config, strat_name = strat_name)
        filtered_means = {k: round(v, 4) for k, v in subres_dict.items() if '_mean' in k}
        print(strat_name, filtered_means)
        for k in res_dict:
            res_dict[k] += subres_dict[k] / num_strat 
for k, v in res_dict.items():
    print(k, v)

# Evaluation Phase 2

Prepare real and synthetic data for evaluations.

In [ ]:
# Load the dataset
from src.evaluations.plot import *
from src.evaluations.evaluations import *
val_set = TensorDataset(real_data_torch)
fake_set = TensorDataset(fake_data_torch)

fake_test_dl = DataLoader(
    fake_set,
    batch_size=config.batch_size,
    shuffle=True
)
real_test_dl = DataLoader(
    val_set,
    batch_size=config.batch_size,
    shuffle=True
)

x_real, x_fake = loader_to_tensor(real_test_dl), loader_to_tensor(fake_test_dl)

In [ ]:
print(next(iter(real_test_dl))[0].shape)
print(next(iter(fake_test_dl))[0].shape)

In [ ]:
for alpha in [0.001, 0.01, 0.05, 0.1]:
    print(f"VaR_{alpha}: fake {calculate_var_es(result[:,4], alpha)[0]:.4f}, real {calculate_var_es(result2[:,4], alpha)[0]:.4f}")
    print(f"ES_{alpha}: fake {calculate_var_es(result[:,4], alpha)[1]:.4f}, real {calculate_var_es(result2[:,4], alpha)[1]:.4f}")

# Visualization

### Samples plot

In [ ]:
plot_samples(real_test_dl, fake_test_dl, config, plot_show = True)

# Stylised facts

### Histogram

In [ ]:
#fig = plot_hists_marginals(x_real, x_fake)

The plot above illustrate the marginal distribution comparisons across various time steps.
Essentially we can quantify this by the marginal distribution loss: 

In [ ]:
# Marginal_loss = to_numpy(HistoLoss(x_real[:, 1:, :], n_bins=50, name='marginal_distribution')(x_fake[:, 1:, :]))
# print('Marginal Loss = ', Marginal_loss)

### Autocorrelation

In [ ]:
# name = 'Our Model'
# compare_acf_matrix(real_test_dl, fake_test_dl, config)

The above plots demonstrated the quality of the generated sample in terms of autocorrelation. Correspondingly, we have the loss

In [ ]:
# acf_loss = to_numpy(ACFLoss(x_real, name='auto_correlation')(x_fake))
# print('autocorrelation Loss = ', acf_loss)

In [ ]:
# # CrossCorrelLoss 인스턴스 생성 시 name 인수를 추가
# cross_corr_loss = CrossCorrelLoss(x_real, max_lag=64, name="cross_correlation")

# # 생성 데이터와 비교하여 손실 계산
# loss_value = cross_corr_loss.compute(x_fake)
# print('Cross-correlation Loss = ', loss_value)


In [ ]:
# # CovLoss 인스턴스 생성
# cov_loss = CovLoss(x_real, name="covariance_loss")

# # 생성 데이터와 비교하여 손실 계산
# loss_value = cov_loss.compute(x_fake)
# print('Covariance Loss = ', loss_value)


### T-sne

In [ ]:
fake_prices_set = TensorDataset(fake_prices)
real_prices_set = TensorDataset(real_prices)

fake_prices_dl = DataLoader(
    fake_prices_set,
    batch_size=config.batch_size,
    shuffle=True
)
real_prices_dl = DataLoader(
    real_prices_set,
    batch_size=config.batch_size,
    shuffle=True
)

tsne_plot(real_prices_dl, fake_prices_dl, config, plot_show =True)

In [ ]:
def log_returns_to_prices(log_returns, s0=100.0):
    cum_log_returns = np.cumsum(log_returns, axis=1)
    prices = s0 * np.exp(cum_log_returns)
    return prices

fake_prices = log_returns_to_prices(fake_data)
real_prices = log_returns_to_prices(real_data)


import torch
import numpy as np
import matplotlib.pyplot as plt
import umap  # pip install umap-learn
import gc

def loader_to_tensor(dl):
    """DataLoader에서 텐서를 빠르게 추출"""
    return torch.cat([x[0] for x in dl], dim=0)

def umap_plot(real_dl, fake_dl, sample_size=1000, save_path=None):
    # 1. 데이터 로드 및 NumPy 변환
    real_X = loader_to_tensor(real_dl).cpu().numpy()
    fake_X = loader_to_tensor(fake_dl).cpu().numpy()

    # 2. 샘플링 사이즈 결정 및 무작위 인덱스 추출
    anal_sample_no = min([sample_size, len(real_X), len(fake_X)])
    
    idx_real = np.random.permutation(len(real_X))[:anal_sample_no]
    idx_fake = np.random.permutation(len(fake_X))[:anal_sample_no]

    ori_data = real_X[idx_real]
    generated_data = fake_X[idx_fake]

    # 3. 데이터 전처리 (벡터화로 Feature 차원 평균 압축)
    prep_data = np.mean(ori_data, axis=2)
    prep_data_hat = np.mean(generated_data, axis=2)

    # UMAP 모델에 넣기 위해 위아래로 병합
    prep_data_final = np.concatenate((prep_data, prep_data_hat), axis=0)

    # 4. UMAP 학습 및 차원 축소
    print(f"UMAP 연산 시작 (총 샘플 수: {2 * anal_sample_no})...")
    # n_neighbors와 min_dist를 조절하여 군집화의 밀도를 튜닝할 수 있습니다.

    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.0001, random_state=42)
    umap_results = reducer.fit_transform(prep_data_final)

    # 5. 고해상도 투명 배경 시각화
    fig, ax = plt.subplots(figsize=(8, 6), dpi=400)
    fig.patch.set_alpha(0.0)
    ax.set_facecolor('none')

    ax.scatter(umap_results[:anal_sample_no, 0], umap_results[:anal_sample_no, 1],
               c='tab:blue', alpha=0.3, label="Historical (Real)", edgecolors='none')
    ax.scatter(umap_results[anal_sample_no:, 0], umap_results[anal_sample_no:, 1],
               c='tab:orange', alpha=0.3, label="Synthetic (Fake)", edgecolors='none')

    ax.legend(loc='best', fontsize=12)
    ax.set_title('UMAP Plot of Time-Series Dynamics', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('UMAP Dimension 1', fontsize=14)
    ax.set_ylabel('UMAP Dimension 2', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(f"{save_path}.png", dpi=400, transparent=True, bbox_inches='tight')
        
    plt.show()

import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. NumPy 배열을 PyTorch Tensor로 변환 (딥러닝 모델 입력용 float32)
fake_prices_tensor = torch.tensor(fake_prices, dtype=torch.float32)
real_prices_tensor = torch.tensor(real_prices, dtype=torch.float32)

# 2. TensorDataset 생성 (변환된 tensor 객체를 넣어야 합니다!)
fake_prices_set = TensorDataset(fake_prices_tensor)
real_prices_set = TensorDataset(real_prices_tensor)

# 3. DataLoader 생성
fake_prices_dl = DataLoader(
    fake_prices_set,
    batch_size=256,
    shuffle=True
)

real_prices_dl = DataLoader(
    real_prices_set,
    batch_size=256,
    shuffle=True
)

umap_plot(real_prices_dl, fake_prices_dl)